# Colab AUDIT — frozen 12-AA harness (config 9162ce44)

Runs the **identical PC harness** on Colab so no AutoAttack queues on the 5070 Ti. Audits a `val_best` (1k→10k), exports per-attack masks, and computes the **paired bootstrap** vs a comparator (LCB).
- **M0** → `paired('M1a','M0')` = **M1a−M0** (Claim-A ✅ CONFIRMED: 10k +2.87pp LCB +0.0234).
- **B1/B2** (ramp family) → audit **R′** (matched control, `RAMP_claimB.py --claimB none`) + B1/B2, then `paired('B1','Rprime')` / `paired('B2','Rprime')` = **Claim B**. (R′, NOT armA_rampfull — see the B1/B2 section.) Plus `collapse(B1,'B1')` for the representation-collapse check.

The setup cell auto-wires a RAMP checkout into `REPO/external/RAMP` (symlinks `/content/RAMP`, else clones) — required for every ramp-family audit.

## Uploads (Drive `MyDrive/attackdro/`)
1. `cifar-10-python.tar.gz` (sha `6d958be0…`) and the **rebuilt** `attackdro_code.zip` (harness + configs + subsets + M1a masks + `collapse_dump.py`).
2. The `val_best` you want to audit (M0 `.pt`; B1/B2/R′ ramp `val_best.pth`).


In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## EDIT ME


In [ ]:
DRIVE = '/content/drive/MyDrive/attackdro'                # everything lives here
DRIVE_CIFAR_TARGZ = f'{DRIVE}/cifar-10-python.tar.gz'
DRIVE_CODE_ZIP    = f'{DRIVE}/attackdro_code.zip'
OUT_BASE          = f'{DRIVE}/audit_out'                  # ALL eval.json + masks + collapse.json write DIRECTLY here (no temp copies)
import os; os.makedirs(OUT_BASE, exist_ok=True)
print('outputs ->', OUT_BASE)


In [ ]:
# deps: the frozen harness needs autoattack (+ torch/torchvision preinstalled)
!pip -q install git+https://github.com/fra31/auto-attack.git 2>&1 | tail -1
import importlib.util; print('autoattack importable:', importlib.util.find_spec('autoattack') is not None)


In [ ]:
# reuse YOUR uploaded CIFAR (hash-checked) -> REPO/data ; unpack code (harness+configs+subsets+M1a masks)
import hashlib, tarfile, zipfile, os, subprocess as _sp
REPO='/content/attackdro'; os.makedirs(REPO, exist_ok=True)
with zipfile.ZipFile(DRIVE_CODE_ZIP) as z: z.extractall(REPO)
h=hashlib.sha256(open(DRIVE_CIFAR_TARGZ,'rb').read()).hexdigest(); assert h.startswith('6d958be074577803'), 'CIFAR mismatch'
os.makedirs(f'{REPO}/data', exist_ok=True)
with tarfile.open(DRIVE_CIFAR_TARGZ) as t: t.extractall(f'{REPO}/data')
assert os.path.exists(f'{REPO}/scripts/eval_multinorm_audit.py') and os.path.exists(f'{REPO}/data/cifar-10-batches-py/test_batch')
assert os.path.exists(f'{REPO}/results/eval/union_bench/M1a/masks_multinorm_v1.npz'), 'M1a masks missing — re-upload the rebuilt zip'

# ramp-family audits (B1/B2/R') need a RAMP checkout at REPO/external/RAMP — the harness ramp_loader
# hardcodes that path. Symlink the existing clone (/content/RAMP) if it's a FULL checkout, else clone fresh.
RAMP_DST=f'{REPO}/external/RAMP'; os.makedirs(f'{REPO}/external', exist_ok=True)
if os.path.exists('/content/RAMP/model_zoo/fast_models.py'):
    _sp.run(['rm','-rf',RAMP_DST]); os.symlink('/content/RAMP', RAMP_DST); print('linked /content/RAMP ->', RAMP_DST)
elif not os.path.exists(f'{RAMP_DST}/model_zoo/fast_models.py'):
    print('cloning RAMP into', RAMP_DST); _sp.run(['git','clone','--depth','1','https://github.com/uiuc-focal-lab/RAMP',RAMP_DST])
assert os.path.exists(f'{RAMP_DST}/model_zoo/fast_models.py'), 'RAMP model_zoo still missing — ramp-family audits will fail'
print('harness + CIFAR + configs + subsets + M1a masks ready at', REPO)
print('RAMP checkout ready:', os.path.realpath(RAMP_DST))


In [ ]:
# audit + collapse + paired-bootstrap helpers. ALL outputs write DIRECTLY to Drive (OUT_BASE) —
# nothing is left only in the ephemeral VM. Harness runs cwd=REPO (for config/subset); --out is a Drive path,
# and the mask sidecar follows --out (repo_path passes absolute paths through), so masks land on Drive too.
import subprocess, numpy as np, json, os
C1K = 'configs/eval/audit_cifar10_preactrn18_multinorm_v3A_testfinal.yaml'
C10K= 'results/eval/union_bench/_config/audit_v3A_test_10k.yaml'
def _dst(name, grade, fn): return f'{OUT_BASE}/{name}/'+('' if grade=='1k' else '10k/')+fn
def run_audit(ckpt, family, name):
    assert os.path.exists(ckpt), f'checkpoint not found: {ckpt}'
    for grade,cfg in [('1k',C1K),('10k',C10K)]:
        out=_dst(name,grade,'eval.json')                       # absolute Drive path
        if os.path.exists(out): print('SKIP',name,grade,'->',out); continue
        os.makedirs(os.path.dirname(out), exist_ok=True)
        cmd=['python','scripts/eval_multinorm_audit.py','--config',cfg,'--checkpoint',ckpt,
             '--model-family',family,'--run-id',f'{name}_{grade}','--checkpoint-role','val_best',
             '--out',out,'--export-masks','--bs','128']
        print('AUDIT',name,grade,'...'); r=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True)
        if not os.path.exists(out):   # audit crashed -> show the REAL traceback, don't mask it
            print('--- STDOUT ---'); print(r.stdout[-1500:])
            print('--- STDERR ---'); print(r.stderr[-2500:])
            raise RuntimeError(f'audit FAILED: {name} {grade} (no eval.json written) — see traceback above')
        d=json.load(open(out)); print(f'  {name} {grade} union={d["full_audit_union"]:.4f}  -> {out}')
def collapse(ckpt, name, n=512):
    """backbone pooled-512 collapse dump (alignment/uniformity/embed_norm); head discarded at eval. Writes to Drive."""
    assert os.path.exists(ckpt), f'checkpoint not found: {ckpt}'
    out=f'{OUT_BASE}/{name}/collapse.json'; os.makedirs(os.path.dirname(out), exist_ok=True)
    env=dict(os.environ, ATTACKDRO_ROOT=REPO, RAMP_DIR=os.path.realpath(f'{REPO}/external/RAMP'))
    cmd=['python','scripts/dev/collapse_dump.py','--checkpoint',ckpt,'--config',C1K,'--name',name,'--n',str(n),'--out',out]
    print('COLLAPSE',name,'...'); r=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True,env=env)
    if not os.path.exists(out):
        print('--- STDOUT ---'); print(r.stdout[-1500:]); print('--- STDERR ---'); print(r.stderr[-2500:])
        raise RuntimeError(f'collapse FAILED: {name}')
    print(r.stdout[-500:]); print('  ->', out)
def _union(name, grade):
    fn='masks_multinorm_v1.npz'
    # prefer Drive; fall back to the zip-baked masks in REPO (e.g. M1a shipped in attackdro_code.zip)
    cands=[_dst(name,grade,fn), f'{REPO}/results/eval/union_bench/{name}/'+('' if grade=='1k' else '10k/')+fn]
    p=next((c for c in cands if os.path.exists(c)), None)
    if p is None: raise FileNotFoundError(f'{name} {grade} masks not found (Drive or zip)')
    d=np.load(p); ns=[k for k in d.files if k!='metadata_json']; u=np.ones(len(d[ns[0]]),bool)
    [u.__iand__(d[n].astype(bool)) for n in ns]; return u
def paired(a,b):
    for grade in ['1k','10k']:
        try: ua,ub=_union(a,grade),_union(b,grade)
        except Exception as e: print(f'{grade}: {a} or {b} masks missing ({e})'); continue
        rng=np.random.default_rng(0); n=len(ua); D=[]
        for _ in range(10000): i=rng.integers(0,n,n); D.append(ua[i].mean()-ub[i].mean())
        print(f'{grade}: union({a})={ua.mean():.4f}  union({b})={ub.mean():.4f}  {a}-{b}={np.mean(D):+.4f}  LCB95={np.quantile(D,0.05):+.4f}  -> {"SIGNIF" if np.quantile(D,0.05)>0 else "CI incl 0"}')
print('helpers ready — outputs write directly to', OUT_BASE)


## M0 → audit + M1a − M0 (Claim-A decisive)


In [ ]:
M0_CKPT = '/content/drive/MyDrive/attackdro/C5_M0_out/ckpt/val_best.pt'   # <- edit to your M0 val_best
run_audit(M0_CKPT, 'robustdro', 'M0')
print('\n=== M1a - M0 (paired bootstrap, LCB) ===')
paired('M1a','M0')   # M1a masks are baked into the zip


## B1 / B2 (ramp family) → B − R′  (baseline pivot: matched control, NOT armA_rampfull)

**Claim B = B1 − R′ / B2 − R′.** R′ = matched RAMP control trained via the SAME patched harness (`RAMP_claimB.py --claimB none`, same lbd5/seed0/at_iter10/80ep/static-lr + same worst-union val_best), differing from B1/B2 *only* by the rep term. The old `armA_rampfull` R used a different script/val-selection/machine → confounded; do **not** use it as the subtraction.

B1's `0.592` val proxy is **discarded as overfit** — its real number is the frozen 12-AA below. The ramp-family setup (RAMP symlink/clone) is handled in the setup cell.

```python
# B1 real number + representation-collapse check
B1 = '/content/drive/MyDrive/attackdro/Bet1_out/B1_pullpush_seed0/val_best.pth'
run_audit(B1, 'ramp', 'B1')     # 1k -> 10k
collapse(B1, 'B1')              # alignment(per-norm)/uniformity/embed_norm, backbone pooled-512

# when R' finishes on Colab-C (upload its val_best.pth to Drive):
RP = '/content/drive/MyDrive/attackdro/Rprime_out/val_best.pth'   # <- edit to your R' path
run_audit(RP, 'ramp', 'Rprime')
paired('B1', 'Rprime')          # ← Claim-B decision number
# B2 (local audit) or after uploading B2 val_best:  run_audit(<B2>, 'ramp', 'B2'); paired('B2','Rprime')
```

Healthy collapse read: `uniformity` clearly negative (features spread), `embed_norm` not tiny, `alignment` moderate (not →1.0 = adv≈clean degeneracy).
